# OCR Model Selection Lab — Colab CPU/GPU

Ce notebook utilise exactement le même moteur que l'application locale et Docker.

1. Dans **Runtime > Change runtime type**, choisissez `CPU` ou `T4 GPU`.
2. Exécutez toutes les cellules dans l'ordre.
3. CUDA accélère EasyOCR lorsqu'un GPU est réellement attribué.
4. Les compteurs de tokens concernent les modèles génératifs qui les exposent ; ils ne s'appliquent pas à EasyOCR.

In [ ]:
# Configuration : utilisez une URL Git accessible, ou laissez vide pour charger un ZIP du projet.
REPO_URL = ""  # Exemple : "https://github.com/organisation/ocr-benchmark.git"
BRANCH = "main"
PROJECT_DIR = "/content/ocr-benchmark"


In [ ]:
import os, pathlib, shutil, subprocess, zipfile

project = pathlib.Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)

if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)
else:
    from google.colab import files
    print("Chargez un ZIP contenant main.py, requirements-ocr.txt, models/, ocr_benchmark/ et dataset/.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if not zip_name:
        raise ValueError("Aucun fichier ZIP reçu.")
    project.mkdir(parents=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(project)
    candidates = [p.parent for p in project.rglob("main.py")]
    if len(candidates) != 1:
        raise RuntimeError(f"Impossible d'identifier la racine du projet : {candidates}")
    project = candidates[0]

os.chdir(project)
print("Project:", project)

In [ ]:
!python -m pip install -q -r requirements-ocr.txt


In [ ]:
import platform, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("Mode CPU actif. Pour tester le GPU : Runtime > Change runtime type > T4 GPU.")

## Test rapide du moteur

Le modèle simulé valide le pipeline sans GPU. Remplacez-le par `easyocr:EasyOCR-Local` pour mesurer EasyOCR sur le matériel Colab.

In [ ]:
from main import load_dataset
from ocr_benchmark.domain import BenchmarkCase
from ocr_benchmark.registry import build_default_registry
from ocr_benchmark.runner import BenchmarkRunner, summarize_results

MODEL_SPEC = "mock:MockOCR-Colab"  # ou "easyocr:EasyOCR-Local"
item = load_dataset()[0]
runner = BenchmarkRunner(build_default_registry())
run_id, results = runner.run([MODEL_SPEC], [BenchmarkCase.from_dict(item)])
display(summarize_results(results))

## Interface complète

Colab affiche une URL publique temporaire. Ne l'utilisez pas avec des documents sensibles.

In [ ]:
from main import APP_CSS, build_ui

app = build_ui()
app.launch(share=True, debug=False, css=APP_CSS)